# FWH directivity (XY) — 2D/3D post-process correction

Based on `plot_directivity_XY_far_near.ipynb`.

The simulation is 2D (periodic z, thickness $L_z$) so the probe measures the **2D line-source** field
($\sim 1/\sqrt{r}$ spreading). The FWH solver uses the **3D Farassat 1A** on the thin $L_z$ z-slab
($\sim 1/r$ spreading), so its raw output is smaller than the probe by the 2D/3D factor.

For a z-uniform source the 2D and 3D (slab) fields are related by a transfer function in the
frequency domain. Using the exact **convected** (uniform-flow Mach $M$) Green's functions:

$$R(f,\theta)=\frac{|G_{2D}|}{L_z\,|G_{3D}|}=\frac{\pi R^*}{\beta L_z}\,\bigl|H_0^{(1)}\bigl(k R^*/\beta^2\bigr)\bigr|$$

with $\beta=\sqrt{1-M^2}$, $k=2\pi f/c$, $R^*=r\sqrt{1-M^2\sin^2\theta}$ (Prandtl–Glauert distance,
compact-source approximation: source at the cylinder centre, $r$ = observer radius).

This script applies $R(f,\theta)$ to the 3D FWH spectrum (per observer) and compares the
**2D-corrected FWH** RMS against the probe RMS. Expect agreement to ~10–20 % (limited by the
compact-source approximation; an exact 2D FWH would need the in-solver 2D formulation).


In [ ]:
import os
import sys
import glob
import math
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
folderVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2_75D",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_40dx_test_3_75D",
]
farNearFolder = [
    ["farfield2", "probe3"],
    ["farfield4", "probe3"],
]
lblVec = [
    "Present LBM Cumulant D/dx=25",
    "Present LBM Cumulant D/dx=40",
]

DPhy = 1.0
DLbVec = [
    40,
    40,
]
rhoPhyVec = [
    1.204,
    1.204,
]
gamma = 1.0
U0PhyVec = [
    68.0,
    68.0,
]
stepRg = [
    [60000, 100000],
    [60000, 100000],
]

# --- 2D/3D post-process correction parameters ---
c_phy = 340.0            # speed of sound [m/s]

# z-domain thickness (periodic length) [m] -> simulation is 2D
LzVec = [
    # 0.025,
    # 0.025,
    0.043,
    0.043,
]

# post-process low-pass cutoff [Hz]: removes aliased LBM numerical modes (4-/8-step)
# that land above the band. Must be > highest physical freq and < lowest alias.
# observer radius r is read per observer from the CSV (all on the r=3 m ring)
fcLowpassVec = [
    150.0,
    150.0,
    150.0,
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

compareFarNear = "far"
# compareFarNear = "near"
# compareFarNear = "both"

dxVec = [DPhy / DLb for DLb in DLbVec]
unitConverterVec = [uc(dx, rhoPhy, gamma) for dx, rhoPhy in zip(dxVec, rhoPhyVec)]
pres0PhyVec = [u.lb_to_phys_pressure(1.0 * u.cs2_lb) for u in unitConverterVec]


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
    coordsStr = header.split('(')[1].split(')')[0].split(',')
    x = float(coordsStr[0]); y = float(coordsStr[1]); z = float(coordsStr[2])
    return x, y, z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

# ---- Bessel J0, Y0 (series; no scipy dependency) ----
def J0(x):
    s = 0.0; term = 1.0; k = 0
    while True:
        s += term
        k += 1
        term *= -(x / 2.0) ** 2 / (k * k)
        if abs(term) < 1e-16:
            break
    return s

def Y0(x):
    if x < 1e-12:
        return -1e15
    g = 0.5772156649
    j0 = J0(x)
    s = 0.0
    x2 = (x / 2.0) ** 2
    Hk = 0.0
    term = 1.0   # (x/2)^{2k}/(k!)^2, k=0
    for k in range(1, 80):
        Hk += 1.0 / k
        term *= x2 / (k * k)
        s += ((-1) ** (k + 1)) * Hk * term
        if abs(Hk * term) < 1e-16:
            break
    return (2.0 / math.pi) * ((math.log(x / 2.0) + g) * j0 + s)

def hankel0_mag(x):
    """|H_0^(1)(x)|"""
    if x < 1e-10:
        return 1e15
    if x >= 8.0:                       # large-argument asymptote
        return math.sqrt(2.0 / (math.pi * x))
    return abs(complex(J0(x), Y0(x)))

def R_2d3d(f, r_obs, theta, M, c, Lz):
    """2D/3D spectral transfer function (convected, compact-source).
       p_2D(f) = p_3D(f) * R(f,theta).  Returns |R| (real, >=0)."""
    if f <= 0.0:
        return 0.0
    beta = math.sqrt(1.0 - M * M)
    Rstar = r_obs * math.sqrt(1.0 - M * M * math.sin(theta) ** 2)   # Prandtl-Glauert distance
    k = 2.0 * math.pi * f / c
    arg = k * Rstar / (beta * beta)
    return (math.pi * Rstar / (beta * Lz)) * hankel0_mag(arg)


In [ ]:
results = []
for case in range(len(folderVec)):
    farFileList = sorted(glob.glob(f"{folderVec[case]}/microphones/{farNearFolder[case][0]}/microphone*.txt"))
    nearFileList = sorted(glob.glob(f"{folderVec[case]}/probes/{farNearFolder[case][1]}/probe*.txt"))
    u = unitConverterVec[case]
    factor_pressure = u.factor_pressure
    M = U0PhyVec[case] / c_phy
    Lz = LzVec[case]

    angles = []
    fwh3d = []
    fwh2d = []
    fwh2d_lp = []
    near = []
    for iFar in range(len(farFileList)):
        x, y, z = get_probe_coords(farFileList[iFar])
        theta = math.atan2(y, x) % (2.0 * math.pi)
        r_obs = math.hypot(x, y)

        # ---- FWH 3D (time-domain RMS, as in the original script) ----
        step = read_col_probe(farFileList[iFar], 0, 2)
        pres = read_col_probe(farFileList[iFar], 2, 2)
        i0 = int(np.abs(step - stepRg[case][0]).argmin())
        i1 = int(np.abs(step - stepRg[case][1]).argmin())
        seg = pres[i0:i1].astype(float)
        seg = seg - np.mean(seg)
        rms3d = np.sqrt(np.mean(seg ** 2))                      # [Pa]

        # ---- FWH 2D-corrected (spectral: multiply by R(f,theta), back to time) ----
        dsp = float(np.median(np.diff(step)))
        dt = u.step_to_time(int(round(dsp))) if dsp > 0 else u.factor_time
        Pf = np.fft.rfft(seg)
        freq = np.fft.rfftfreq(len(seg), d=dt)
        R = np.array([R_2d3d(f, r_obs, theta, M, c_phy, Lz) for f in freq])
        seg2d = np.fft.irfft(Pf * R, n=len(seg))
        rms2d = np.sqrt(np.mean(seg2d ** 2))                    # [Pa]

        # ---- low-pass to remove aliased LBM numerical modes (4-step/8-step) ----
        # these modes sit above the output Nyquist, so they alias down into the band;
        # since the physical tone is far below, a brickwall low-pass on the FWH output
        # spectrum cleanly removes them without touching the tone.
        lpMask = (freq <= fcLowpassVec[case]).astype(float)
        seg2d_lp = np.fft.irfft(Pf * R * lpMask, n=len(seg))
        rms2d_lp = np.sqrt(np.mean(seg2d_lp ** 2))              # [Pa]

        if compareFarNear == "near" or compareFarNear == "both":
            # ---- probe (2D simulation, density -> LB pressure) ----
            pstep = read_col_probe(nearFileList[iFar], 0, 2)
            rho = read_col_probe(nearFileList[iFar], 2, 2)
            pi0 = int(np.abs(pstep - stepRg[case][0]).argmin())
            pi1 = int(np.abs(pstep - stepRg[case][1]).argmin())
            # dp = (rho[pi0:pi1].astype(float) - 1.0) * u.cs2_lb
            dp = rho[pi0:pi1].astype(float) * u.cs2_lb
            dp = dp - np.mean(dp)
            rms_near = np.sqrt(np.mean(dp ** 2))                    # [LB]

        angles.append(theta)
        # fwh3d.append(rms3d / factor_pressure)                   # [LB]
        # fwh2d.append(rms2d / factor_pressure)                   # [LB]
        # fwh2d_lp.append(rms2d_lp / factor_pressure)             # [LB]
        fwh3d.append(rms3d / factor_pressure / u.cs2_lb)        # [LB] normalized
        fwh2d.append(rms2d / factor_pressure / u.cs2_lb)        # [LB] normalized
        fwh2d_lp.append(rms2d_lp / factor_pressure / u.cs2_lb)  # [LB] normalized
        if compareFarNear == "near" or compareFarNear == "both":
            # near.append(rms_near)                                   # [LB]
            near.append(rms_near / u.cs2_lb)                        # [LB] normalized

    res = {
        'angles': np.array(angles),
        'fwh3d': np.array(fwh3d),
        'fwh2d': np.array(fwh2d),
        'fwh2d_lp': np.array(fwh2d_lp),
        'near': np.array(near),
    }
    results.append(res)
    if compareFarNear == "near" or compareFarNear == "both":
        med3d = np.median(res['near'] / np.maximum(res['fwh3d'], 1e-30))
        med2d = np.median(res['near'] / np.maximum(res['fwh2d'], 1e-30))
        print(f"case {case}: M={M:.3f}, Lz={Lz}, N_obs={len(angles)}")
        print(f"   median probe/FWH(3D)    = {med3d:.1f}   (the 2D/3D spreading gap)")
        print(f"   median probe/FWH(2Dcorr)= {med2d:.2f}   (should be ~1 if correction works)")


In [ ]:
# ref1File = "../ref/directivity_rmsPTilde_inoue_Ma02.csv"
ref1File = "../ref/directivity_rmsPTilde_modified_inoue_Ma02.csv"
ref1Angles = read_csv_col(ref1File, 1, 1)
ref1PFluct = read_csv_col(ref1File, 0, 1)
ref1Radians = np.radians(ref1Angles)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8), facecolor='w', subplot_kw={'projection': 'polar'})

ax.plot(ref1Radians, ref1PFluct, marker='o', fillstyle='none', mew=2, lw=0, c='k', label="Inoue & Hatakeyama, DNS (D/dx=200)")

for case in range(len(folderVec)):
    r = results[case]
    angle = 58
    degArray = np.degrees(r['angles'])
    mask = ~((degArray <= angle) | (degArray >= (360-angle)))

    # ax.plot(r['angles'], r['fwh3d'], '--', lw=1, label=f"{lblVec[case]} FWH 3D (uncorrected)")
    # ax.plot(r['angles'], r['fwh2d'], '-',  lw=1, alpha=0.5, label=f"{lblVec[case]} FWH 2D (raw, noisy)")
    ax.plot(r['angles'], r['fwh2d_lp'], '-',  lw=2, label=f"{lblVec[case]} FWH 2D + low-pass(<{fcLowpassVec[case]:.0f}Hz)", c=colorVec[case])
    if compareFarNear == "near" or compareFarNear == "both":
        ax.plot(r['angles'][mask], r['near'][mask], ':',  lw=2, label=f"{lblVec[case]} probe (2D sim)", c=colorVec[case])
ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10), fontsize=9, frameon=False)
# plt.tight_layout()
# plt.show()
ax.set_ylim(0, 1E-4)
lines, labels = ax.set_thetagrids(np.arange(0, 360, 30))
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), frameon=False)

ticks = np.linspace(0, 1E-4, 5)
ax.set_rticks(ticks)
labelOuter = r"$1.0 \times 10^{-4}$" + "\n" + r"$p\prime_{RMS}/(\rho_0c_s^2)$"
labels = [
    "0.0",
    "",
    "0.5",
    "",
    labelOuter
]
ax.set_yticklabels(labels)
ax.set_rlabel_position(15)

In [ ]:
# Spectrum at the observer closest to 90 deg (lift direction): FWH 3D, FWH 2D-corrected, probe
case = 0
u = unitConverterVec[case]
factor_pressure = u.factor_pressure
M = U0PhyVec[case] / c_phy
Lz = LzVec[case]
r = results[case]

idx90 = int(np.abs(r['angles'] - math.pi / 2).argmin())
farFileList = sorted(glob.glob(f"{folderVec[case]}/microphones/{farNearFolder[case][0]}/microphone*.txt"))
nearFileList = sorted(glob.glob(f"{folderVec[case]}/probes/{farNearFolder[case][1]}/probe*.txt"))

x, y, z = get_probe_coords(farFileList[idx90])
theta = math.atan2(y, x) % (2 * math.pi)
r_obs = math.hypot(x, y)
step = read_col_probe(farFileList[idx90], 0, 2)
pres = read_col_probe(farFileList[idx90], 2, 2)
i0 = int(np.abs(step - stepRg[case][0]).argmin())
i1 = int(np.abs(step - stepRg[case][1]).argmin())
seg = pres[i0:i1].astype(float)
seg = seg - np.mean(seg)
dt = u.step_to_time(int(round(float(np.median(np.diff(step))))))
Pf = np.fft.rfft(seg)
freq = np.fft.rfftfreq(len(seg), d=dt)
R = np.array([R_2d3d(f, r_obs, theta, M, c_phy, Lz) for f in freq])

if compareFarNear == "near" or compareFarNear == "both":
    # probe spectrum in Pa
    pstep = read_col_probe(nearFileList[idx90], 0, 2)
    rho = read_col_probe(nearFileList[idx90], 2, 2)
    pi0 = int(np.abs(pstep - stepRg[case][0]).argmin())
    pi1 = int(np.abs(pstep - stepRg[case][1]).argmin())
    dppa = (rho[pi0:pi1].astype(float) - 1.0) * u.cs2_lb * factor_pressure
    dppa = dppa - np.mean(dppa)
    Pn = np.fft.rfft(dppa)
    fn = np.fft.rfftfreq(len(dppa), d=dt)

fig, ax = plt.subplots(1, 1, figsize=(9, 5), facecolor='w')
fcLp = fcLowpassVec[case]
lpMask90 = (freq <= fcLp).astype(float)
m = freq <= max(200.0, 1.0 / (2 * dt))   # show up to Nyquist so the alias is visible
ax.semilogy(freq[m], np.abs(Pf[m]) / len(seg) * 2, '-',  label='FWH 3D (uncorrected, raw)')
ax.semilogy(freq[m], np.abs(Pf[m]) * R[m] / len(seg) * 2, '-', alpha=0.4, label='FWH 2D-corrected (raw, with alias)')
ax.semilogy(freq[m], np.abs(Pf[m]) * R[m] * lpMask90[m] / len(seg) * 2, '-', lw=2, label=f'FWH 2D + low-pass(<{fcLp:.0f}Hz)')
ax.axvline(fcLp, ls=':', c='gray', label=f'low-pass cutoff {fcLp:.0f}Hz')
if compareFarNear == "near" or compareFarNear == "both":
    ax.semilogy(fn[fn <= max(200.0, 1.0 / (2 * dt))], np.abs(Pn[fn <= max(200.0, 1.0 / (2 * dt))]) / len(dppa) * 2, '--', label='probe (2D sim)')
ax.set_xlabel('frequency [Hz]')
ax.set_ylabel('|P| [Pa]')
ax.set_title(f'spectrum @ ~90 deg (r={r_obs:.1f} m, M={M:.2f})')
ax.legend(fontsize=9)
ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
plt.show()


## Notes

- **Uncorrected FWH (3D)** is ~200x below the probe — the 2D/3D spreading gap
  (theory `(pi*r/Lz)*|H0(kr)|` ~ 208 for r=3 m, Lz=0.04 m, f=13.25 Hz, M=0.2).
- **2D-corrected FWH** should land on top of the probe (within ~10-20 %). Residual error comes
  from the compact-source approximation (source extent D/r ~ 0.33, but R varies only ~10-15 %
  across the source) and from any 3D-amplitude convention difference (1/R* vs the solver's 1/r_ret).
- The correction is frequency-dependent, so it works for broadband too (not just the tonal peak).
- DC (f=0) is left uncorrected (H0 is singular there) — it is removed by the mean subtraction anyway.
- For an exact match (no compact-source approximation) the 2D Green's function must be applied
  per source point inside the FWH integral — i.e. a full in-solver 2D FWH formulation.
